# 🌍 Week 8 — Advanced Spatial Analytics
### Geospatial Python Mastery | Module 4: Spatial Databases
---
| | |
|---|---|
| **Module** | Module 4: Spatial Databases |
| **Week** | 8 of 10 |
| **Duration** | 3 hours |
| **Level** | Advanced |
| **Prerequisites** | Weeks 1–7 (Python, GeoPandas, PostgreSQL/PostGIS basics) |

> This week takes your PostGIS skills to production level. You'll master spatial joins at scale, nearest-neighbour queries, geometry validity checking and repair, query optimisation with VACUUM/indexes, and dissolve aggregations.

### 📋 Table of Contents
| # | Section |
|---|---------|
| 1 | Spatial Joins at Scale |
| 2 | Nearest-Neighbour Queries |
| 3 | Geometry Validity Checking and Repair |
| 4 | Performance Tuning: VACUUM, Indexes, Clustering |
| 5 | Aggregation: ST_Union, ST_Collect, ST_ConvexHull |
| 6 | Optimising Slow Spatial Queries |
| 7 | Spatial Aggregation Patterns |
| 8 | Working with Large Datasets |
| 9 | Putting It Together: A Spatial Analysis Pipeline |
| 10 | Mini-Lab — Optimise and Analyse |

### 🔣 Symbol Guide
| Symbol | Meaning |
|--------|---------|
| 💻 | Code walkthrough |
| 🎯 | Exercise |
| ✅ | Solution |
| 🔬 | Lab step |
| 🐘 | PostgreSQL/PostGIS-specific |
| 🔷 | Demo-safe (Shapely + SQLite / pure Python) |
| 📖 | Concept explanation |

*Keyboard shortcuts: `Shift+Enter` runs a cell, `A` inserts above, `B` inserts below, `Ctrl+/` toggles comments.*

## 🎯 Learning Objectives

| # | Objective |
|---|-----------|
| 1 | Explain how spatial joins differ from attribute joins and when GiST indexes matter |
| 2 | Use GeoPandas sjoin() with within, intersects, and other spatial predicates |
| 3 | Apply KNN thinking with STRtree in Python and <-> in PostGIS |
| 4 | Detect invalid geometries and interpret explain_validity / ST_IsValidReason output |
| 5 | Repair broken polygons with make_valid, ST_MakeValid, and buffer(0) safely |
| 6 | Improve planner performance with VACUUM ANALYZE, partial indexes, and CLUSTER |
| 7 | Choose correctly between ST_Union, ST_Collect, ST_ConvexHull, and ST_Envelope |
| 8 | Rewrite slow spatial filters so ST_DWithin and indexes can accelerate them |
| 9 | Process larger datasets with chunking, bbox filtering, and geometry simplification |
| 10 | Build an end-to-end spatial analysis pipeline from validation to export |

**How to use this notebook:**
- Run the notebook from top to bottom the first time so shared GeoDataFrames are available later.
- Treat 🔷 cells as safe local demos and 🐘 cells as production SQL patterns to copy into PostGIS.
- Pause after each exercise to solve it yourself before revealing the ✅ solution cell.
- Keep CRS awareness front-and-centre: use EPSG:4326 for storage/interchange and EPSG:28992 for Dutch metric analysis.

In [ ]:
# ── Environment detection and package installation ────────────────────────────
import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

REQUIRED = [
    'geopandas',
    'shapely',
    'pyproj',
    'sqlalchemy',
    'pandas',
    'matplotlib',
    'folium',
    'numpy',
]

if IN_COLAB:
    for pkg in REQUIRED:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
    print('✅ Packages installed (Colab)')
else:
    print('ℹ️  Running locally — ensure your environment has all required packages.')
    print('   pip install', ' '.join(REQUIRED))

# Windows users: if pyproj/geopandas fail to import, install via conda:
#   conda install -c conda-forge geopandas pyproj shapely

---
## 📖 Section 1 — Spatial Joins at Scale

A **spatial join** combines two datasets using a spatial relationship instead of a shared text or numeric key. 
Attribute joins match `id = id`; spatial joins ask questions like “which district contains this POI?” or “which buildings intersect this flood zone?”.

| Predicate | Meaning | Typical use |
|-----------|---------|-------------|
| `intersects` | Features share any space | Overlay, clipping, collision checks |
| `within` | Left geometry is completely inside right geometry | Point-in-polygon, zoning checks |
| `contains` | Right geometry is inside left geometry | Administrative boundary queries |
| `dwithin` | Features are within a threshold distance | Nearby search, service areas |

Spatial joins become expensive fast because every candidate pair could need a geometry test. PostGIS accelerates this with **GiST indexes**, which use bounding boxes to reject most pairs before exact predicates run.

In [ ]:
# 💻 1.2  GeoPandas spatial join walkthrough (🔷)
# ─────────────────────────────────────────────────────────────────────────────
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, Polygon
import numpy as np

# Create 5 district polygons (bounding boxes)
districts_data = {
    'name': ['North', 'South', 'East', 'West', 'Centre'],
    'population': [45000, 62000, 38000, 51000, 89000],
    'geometry': [
        Polygon([(4.85,52.40),(4.95,52.40),(4.95,52.47),(4.85,52.47),(4.85,52.40)]),
        Polygon([(4.85,52.33),(4.95,52.33),(4.95,52.40),(4.85,52.40),(4.85,52.33)]),
        Polygon([(4.95,52.35),(5.05,52.35),(5.05,52.45),(4.95,52.45),(4.95,52.35)]),
        Polygon([(4.75,52.35),(4.85,52.35),(4.85,52.45),(4.75,52.45),(4.75,52.35)]),
        Polygon([(4.87,52.36),(4.93,52.36),(4.93,52.42),(4.87,52.42),(4.87,52.36)]),
    ]
}
districts = gpd.GeoDataFrame(districts_data, crs='EPSG:4326')

# Create 20 random POI points
np.random.seed(42)
pois_data = {
    'poi_name': [f'POI_{i:02d}' for i in range(20)],
    'category': np.random.choice(['cafe','park','school','hospital'], 20),
    'geometry': [Point(np.random.uniform(4.76,5.04), np.random.uniform(52.34,52.46)) for _ in range(20)]
}
pois = gpd.GeoDataFrame(pois_data, crs='EPSG:4326')

# Spatial join: which district contains each POI?
joined = gpd.sjoin(pois, districts, how='left', predicate='within')
print(joined[['poi_name','category','name','population']].to_string())
print(f"\nTotal POIs: {len(pois)}  |  Joined: {joined['name'].notna().sum()}  |  Unmatched: {joined['name'].isna().sum()}")

In [ ]:
# 💻 1.3  PostGIS spatial join patterns (🐘)
# ─────────────────────────────────────────────────────────────────────────────
sql_lines = [
    "-- Spatial join: tag each POI with the district it falls within",
    "SELECT p.poi_name, p.category, d.name AS district, d.population",
    "FROM pois p",
    "LEFT JOIN districts d ON ST_Within(p.geom, d.geom);",
    "",
    "-- LATERAL nearest-neighbor join (KNN)",
    "SELECT p.poi_name, d.name AS nearest_district,",
    "       ST_Distance(ST_Transform(p.geom,28992), ST_Transform(d.centroid,28992)) AS dist_m",
    "FROM pois p",
    "CROSS JOIN LATERAL (",
    "    SELECT name, ST_Centroid(geom) AS centroid",
    "    FROM districts",
    "    ORDER BY p.geom <-> geom",
    "    LIMIT 1",
    ") d;",
    "",
    "-- Anti-join: POIs not within any district",
    "SELECT p.poi_name",
    "FROM pois p",
    "WHERE NOT EXISTS (",
    "    SELECT 1 FROM districts d WHERE ST_Within(p.geom, d.geom)",
    ");",
]
print("\n".join(sql_lines))

In [ ]:
# 💻 1.4  Join coverage by district and category (🔷)
# ─────────────────────────────────────────────────────────────────────────────
district_counts = joined.groupby(["name", "category"]).size().unstack(fill_value=0)
district_counts = district_counts.reindex(districts["name"], fill_value=0)
district_counts["total"] = district_counts.sum(axis=1)
print(district_counts.to_string())

In [ ]:
# 💻 1.5  Predicate cheat sheet and index intuition (🔷)
# ─────────────────────────────────────────────────────────────────────────────
predicate_table = pd.DataFrame(
    [
        ('within', 'point inside polygon', 'Excellent with indexed polygon bbox prefilter'),
        ('contains', 'polygon owns the interior point', 'Equivalent topological inverse of within'),
        ('intersects', 'any overlap at all', 'Common overlay predicate; still benefits from GiST'),
        ('dwithin', 'distance threshold', 'Use for proximity search rather than distance < N'),
    ],
    columns=['Predicate', 'Meaning', 'Performance note'],
)
print(predicate_table.to_string(index=False))

### 🎯 Exercise 1 — Count POIs per category inside each district

**Task:** Using the GeoDataFrames from 1.2, perform a spatial join and print a pivot table showing category counts by district.

**Steps:**
1. Join `pois` to `districts` with `gpd.sjoin(..., predicate="within")`.
2. Group by district name and category.
3. Reshape the result into a pivot table with `unstack(fill_value=0)`.

**Hint:**

```python
joined = gpd.sjoin(pois, districts[['name','geometry']], how='left', predicate='within')
```

In [ ]:
# 🎯 Exercise 1 — your code here ────────────────────────────────
# 1. Perform the spatial join.
# 2. Group by district and category.
# 3. Print the pivot table.

In [ ]:
# ✅ Exercise 1 — Solution ──────────────────────────────────────
joined_ex1 = gpd.sjoin(pois, districts[['name','geometry']], how='left', predicate='within')
pivot = joined_ex1.groupby(['name', 'category']).size().unstack(fill_value=0)
pivot = pivot.reindex(districts["name"], fill_value=0)
print(pivot.to_string())

### 📖 Section 1 takeaway

- Spatial joins feel simple at notebook scale, but production performance depends heavily on how quickly the engine can cut down the candidate pairs.
- In PostGIS, GiST indexes and index-aware predicates are what make `JOIN ... ON ST_Within(...)` practical on large tables.

---
## 📖 Section 2 — Nearest-Neighbour Queries

KNN means **K-nearest neighbours**: find the closest features to a reference geometry. In PostGIS the `<->` operator is special because `ORDER BY geom <-> ref_geom LIMIT k` can use the GiST index.

| Pattern | Best use | Notes |
|---------|----------|-------|
| `<->` | Order by nearest | Index-assisted approximate ordering, then exact distance output |
| `ST_Distance` | Exact measurement | Great for final reporting, not ideal alone in `WHERE` |
| `ST_DWithin` | Distance threshold filter | Index-friendly candidate selection before detailed ranking |

The usual production recipe is: use the index to narrow candidates, then compute exact metric distance in a projected CRS.

In [ ]:
# 💻 2.2  STRtree nearest-neighbour demo (🔷)
# ─────────────────────────────────────────────────────────────────────────────
from shapely.strtree import STRtree
from shapely.geometry import Point
import numpy as np

np.random.seed(7)
all_points = [Point(np.random.uniform(4.8,5.0), np.random.uniform(52.3,52.5)) for _ in range(500)]
labels = [f'Feature_{i}' for i in range(500)]

query_point = Point(4.9007, 52.3789)  # Amsterdam Centraal
k = 5

tree = STRtree(all_points)

# KNN using STRtree.nearest (Shapely 2.x)
try:
    indices = tree.query(query_point.buffer(0.1))  # candidates within 0.1 deg
    # Sort by exact distance
    candidates = [(i, all_points[i].distance(query_point)) for i in indices]
    candidates.sort(key=lambda x: x[1])
    knn = candidates[:k]
    print(f"Top-{k} nearest to Amsterdam Centraal:")
    for rank, (idx, dist) in enumerate(knn, 1):
        dist_m = dist * 111_000
        print(f"  {rank}. {labels[idx]}  dist={dist_m:.0f} m")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# 💻 2.3  PostGIS KNN SQL with <-> (🐘)
# ─────────────────────────────────────────────────────────────────────────────
sql_lines = [
    "-- 5 nearest POIs to Amsterdam Centraal using <-> operator",
    "SELECT",
    "    poi_name,",
    "    category,",
    "    ST_Distance(",
    "        ST_Transform(geom, 28992),",
    "        ST_Transform(ST_SetSRID(ST_MakePoint(4.9007,52.3789),4326), 28992)",
    "    ) AS dist_m",
    "FROM pois",
    "ORDER BY geom <-> ST_SetSRID(ST_MakePoint(4.9007,52.3789), 4326)",
    "LIMIT 5;",
    "",
    "-- For each district centroid, find its 3 nearest POIs",
    "SELECT d.name AS district, p.poi_name, p.category,",
    "       ST_Distance(ST_Transform(p.geom,28992), ST_Transform(d.centroid,28992)) AS dist_m",
    "FROM districts d",
    "CROSS JOIN LATERAL (",
    "    SELECT poi_name, category, geom",
    "    FROM pois",
    "    ORDER BY geom <-> d.centroid",
    "    LIMIT 3",
    ") p;",
]
print("\n".join(sql_lines))

In [ ]:
# 💻 2.4  Brute-force versus indexed candidate search (🔷)
# ─────────────────────────────────────────────────────────────────────────────
dist_all = [(labels[i], pt.distance(query_point) * 111_000) for i, pt in enumerate(all_points)]
dist_all.sort(key=lambda x: x[1])
print("Brute-force top 5:")
for rank, (label, dist_m) in enumerate(dist_all[:5], start=1):
    print(f"  {rank}. {label:<12} {dist_m:8.0f} m")

print("\nCandidate-window top 5 from STRtree workflow:")
for rank, (idx, dist) in enumerate(knn, start=1):
    print(f"  {rank}. {labels[idx]:<12} {dist * 111_000:8.0f} m")

In [ ]:
# 💻 2.5  Nearest POI to each district centroid preview (🔷)
# ─────────────────────────────────────────────────────────────────────────────
district_centroids = districts.copy()
district_centroids["centroid"] = district_centroids.geometry.centroid
preview_rows = []
for _, row in district_centroids.iterrows():
    dists = pois.geometry.distance(row["centroid"])
    idx = dists.idxmin()
    preview_rows.append({
        "district": row["name"],
        "nearest_poi": pois.loc[idx, "poi_name"],
        "category": pois.loc[idx, "category"],
        "dist_deg": round(float(dists.loc[idx]), 5),
    })
print(pd.DataFrame(preview_rows).to_string(index=False))

### 🎯 Exercise 2 — Find the 3 nearest POIs to each district centroid

**Task:** From the `pois` GeoDataFrame, find the 3 nearest POIs to each district centroid using a Python loop + Shapely distance. Print the result as a table.

**Steps:**
1. Project both datasets to EPSG:28992 so distance is measured in metres.
2. Loop over district centroids and compute distances to all POIs.
3. Sort, keep the top 3, and collect rows for display.

**Hint:**

```python
pois_rd = pois.to_crs(28992); districts_rd = districts.to_crs(28992)
```

In [ ]:
# 🎯 Exercise 2 — your code here ────────────────────────────────
# 1. Project districts and POIs to EPSG:28992.
# 2. Loop over district centroids.
# 3. Sort by distance and print the top 3 per district.

In [ ]:
# ✅ Exercise 2 — Solution ──────────────────────────────────────
pois_rd = pois.to_crs(28992)
districts_rd = districts.to_crs(28992)
rows = []
for _, drow in districts_rd.iterrows():
    centroid = drow.geometry.centroid
    dists = pois_rd.geometry.distance(centroid)
    nearest_idx = dists.nsmallest(3).index
    for rank, idx in enumerate(nearest_idx, start=1):
        rows.append({
            "district": drow["name"],
            "rank": rank,
            "poi_name": pois.loc[idx, "poi_name"],
            "category": pois.loc[idx, "category"],
            "dist_m": round(float(dists.loc[idx]), 1),
        })
nearest_table = pd.DataFrame(rows)
print(nearest_table.to_string(index=False))

### 📖 Section 2 takeaway

- Use `<->` or an index-backed search structure to discover candidates quickly, then calculate exact distances only for the short list.
- In SQL, `ORDER BY geom <-> ref LIMIT k` is often the cleanest way to express nearest-neighbour search.

---
## 📖 Section 3 — Geometry Validity Checking and Repair

Invalid geometry can break overlays, dissolve operations, and spatial indexes. Common causes include self-intersecting rings, duplicate points, wrong ring direction, and unclosed boundaries.

| Problem | Example symptom | Typical fix |
|---------|-----------------|-------------|
| Self-intersection | Bowtie polygon, figure-8 ring | `make_valid` / `ST_MakeValid` |
| Duplicate vertices | Extra repeated coordinates | Clean geometry, then validate again |
| Unclosed ring | First and last coordinate differ | Rebuild polygon ring correctly |
| Mixed topology issues | Overlay crashes or empty results | Validate before joins/union |

PostGIS gives you `ST_IsValid`, `ST_IsValidReason`, and `ST_MakeValid`. In Python, Shapely mirrors the same workflow with `is_valid`, `explain_validity`, and `make_valid`.

In [ ]:
# 💻 3.2  Create invalid geometries and repair them (🔷)
# ─────────────────────────────────────────────────────────────────────────────
from shapely.geometry import Polygon, MultiPolygon
from shapely.validation import explain_validity, make_valid

# Self-intersecting "bowtie" polygon (figure-8)
bowtie = Polygon([(0,0),(2,2),(2,0),(0,2),(0,0)])
print(f"Bowtie valid    : {bowtie.is_valid}")
print(f"Reason          : {explain_validity(bowtie)}")

# Repair with make_valid
fixed = make_valid(bowtie)
print(f"Fixed type      : {fixed.geom_type}")
print(f"Fixed valid     : {fixed.is_valid}")
print(f"Fixed area      : {fixed.area:.4f}")

# Buffer(0) trick — classic repair
buffered = bowtie.buffer(0)
print(f"\nBuffer(0) type  : {buffered.geom_type}")
print(f"Buffer(0) valid : {buffered.is_valid}")

# Validate a whole GeoDataFrame column
import geopandas as gpd
from shapely.geometry import Point
gdf_test = gpd.GeoDataFrame(
    {'name': ['ok','bowtie','point']},
    geometry=[Polygon([(0,0),(1,0),(1,1),(0,1),(0,0)]), bowtie, Point(1,1)],
    crs='EPSG:4326'
)
gdf_test['is_valid'] = gdf_test.geometry.is_valid
gdf_test['reason'] = gdf_test.geometry.apply(explain_validity)
print(gdf_test[['name','is_valid','reason']].to_string())

In [ ]:
# 💻 3.3  PostGIS validity audit and repair SQL (🐘)
# ─────────────────────────────────────────────────────────────────────────────
sql_lines = [
    "-- Find all invalid geometries",
    "SELECT id, name, ST_IsValid(geom) AS valid, ST_IsValidReason(geom) AS reason",
    "FROM urban_features",
    "WHERE NOT ST_IsValid(geom);",
    "",
    "-- Repair with ST_MakeValid (PostGIS 2.4+)",
    "UPDATE urban_features",
    "SET geom = ST_MakeValid(geom)",
    "WHERE NOT ST_IsValid(geom);",
    "",
    "-- Verify repair",
    "SELECT COUNT(*) AS still_invalid",
    "FROM urban_features",
    "WHERE NOT ST_IsValid(geom);",
    "",
    "-- Alternative: ST_Buffer(geom, 0) repair",
    "UPDATE urban_features",
    "SET geom = ST_Buffer(geom, 0)",
    "WHERE NOT ST_IsValid(geom) AND ST_GeometryType(geom) = 'ST_Polygon';",
]
print("\n".join(sql_lines))

In [ ]:
# 💻 3.4  Compare make_valid and buffer(0) outputs (🔷)
# ─────────────────────────────────────────────────────────────────────────────
repair_compare = pd.DataFrame(
    [
        ("original", bowtie.geom_type, bowtie.is_valid, round(bowtie.area, 4)),
        ("make_valid", fixed.geom_type, fixed.is_valid, round(fixed.area, 4)),
        ("buffer(0)", buffered.geom_type, buffered.is_valid, round(buffered.area, 4)),
    ],
    columns=['method', 'geom_type', 'is_valid', 'area'],
)
print(repair_compare.to_string(index=False))

In [ ]:
# 💻 3.5  Batch WKT validity audit recipe (🔷)
# ─────────────────────────────────────────────────────────────────────────────
from shapely import wkt

wkts = [
    'POLYGON((0 0, 2 0, 2 2, 0 2, 0 0))',
    'POLYGON((0 0, 2 2, 2 0, 0 2, 0 0))',
    'LINESTRING(0 0, 1 1, 2 2)',
]
rows = []
for txt in wkts:
    geom = wkt.loads(txt)
    rows.append({
        "wkt": txt,
        "geom_type": geom.geom_type,
        "is_valid": geom.is_valid,
        "reason": explain_validity(geom),
    })
print(pd.DataFrame(rows).to_string(index=False))

### 🎯 Exercise 3 — Validate and repair a mixed WKT list

**Task:** Given a list of six WKT strings, validate each geometry, repair the invalid ones, and print a before/after table.

**Steps:**
1. Parse the WKT strings with `shapely.wkt.loads`.
2. Use `explain_validity` to record the reason for each geometry.
3. Repair invalid geometries with `make_valid` and compare geometry type + validity after repair.

**Hint:**

```python
from shapely import wkt
from shapely.validation import explain_validity, make_valid
```

In [ ]:
# 🎯 Exercise 3 — your code here ────────────────────────────────
# 1. Create a list of 6 WKT strings.
# 2. Load each geometry and record validity info.
# 3. Repair invalid geometries and print a comparison table.

In [ ]:
# ✅ Exercise 3 — Solution ──────────────────────────────────────
wkt_list = [
    'POLYGON((0 0, 2 0, 2 2, 0 2, 0 0))',
    'POLYGON((0 0, 2 2, 2 0, 0 2, 0 0))',
    'POLYGON((0 0, 1 0, 1 1, 0 1, 0 1, 0 0))',
    'POLYGON((3 0, 5 0, 5 2, 3 2, 3 0))',
    'POINT(1 1)',
    'LINESTRING(0 0, 1 1, 2 1)',
]
rows = []
for txt in wkt_list:
    geom = wkt.loads(txt)
    repaired = make_valid(geom) if not geom.is_valid else geom
    rows.append({
        "wkt": txt,
        "before_valid": geom.is_valid,
        "reason": explain_validity(geom),
        "after_type": repaired.geom_type,
        "after_valid": repaired.is_valid,
    })
print(pd.DataFrame(rows).to_string(index=False))

### 📖 Section 3 takeaway

- Validate before you join, union, or index. Broken polygons can silently poison downstream analysis.
- `buffer(0)` is a classic shortcut, but `make_valid` / `ST_MakeValid` is more explicit and usually safer for production pipelines.

---
## 📖 Section 4 — Performance Tuning: VACUUM, Indexes, Clustering

Spatial performance is not only about query text. PostgreSQL also needs healthy table statistics and well-targeted indexes.

| Tool | What it does | When to use it |
|------|---------------|----------------|
| `VACUUM ANALYZE` | Reclaims dead tuples and refreshes planner statistics | After bulk loads, major updates, or deletes |
| Partial index | Indexes only rows that match a `WHERE` clause | When many rows are irrelevant to common queries |
| `CLUSTER` | Physically reorders the table based on an index | When locality matters and range scans are frequent |

The planner can only choose a good path when table statistics and index design reflect the data you actually query.

In [ ]:
# 💻 4.2  Maintenance SQL patterns (🐘)
# ─────────────────────────────────────────────────────────────────────────────
sql_lines = [
    "-- Update statistics after bulk load",
    "VACUUM ANALYZE urban_features;",
    "",
    "-- Partial index: only index valid geometries",
    "CREATE INDEX uf_valid_geom_idx ON urban_features USING GIST(geom)",
    "WHERE is_valid = TRUE;",
    "",
    "-- Cluster table by spatial index (one-time, expensive — but fast queries after)",
    "CLUSTER urban_features USING uf_valid_geom_idx;",
    "ANALYZE urban_features;",
    "",
    "-- Check index usage with pg_stat_user_indexes",
    "SELECT indexrelname, idx_scan, idx_tup_read, idx_tup_fetch",
    "FROM pg_stat_user_indexes",
    "WHERE relname = 'urban_features'",
    "ORDER BY idx_scan DESC;",
]
print("\n".join(sql_lines))

In [ ]:
# 💻 4.3  Index-like speedup with STRtree (🔷)
# ─────────────────────────────────────────────────────────────────────────────
import time
import numpy as np
from shapely.geometry import Point
from shapely.strtree import STRtree

np.random.seed(99)
N = 50_000
points = [Point(np.random.uniform(4.7,5.1), np.random.uniform(52.2,52.5)) for _ in range(N)]
query  = Point(4.9, 52.37).buffer(0.02)

# Without index
t0 = time.perf_counter()
brute = [p for p in points if p.within(query)]
t_brute = time.perf_counter() - t0

# With STRtree
tree = STRtree(points)
t0 = time.perf_counter()
tree_hits = list(tree.query(query))
t_tree = time.perf_counter() - t0

print(f"Dataset size  : {N:,} points")
print(f"Brute force   : {len(brute):4d} hits in {t_brute*1000:.1f} ms")
print(f"STRtree       : {len(tree_hits):4d} hits in {t_tree*1000:.1f} ms")
print(f"Speedup       : {t_brute/max(t_tree,0.001):.0f}x")
print("(In PostGIS: CLUSTER + GiST gives similar or greater benefit on large tables)")

In [ ]:
# 💻 4.4  When to reach for each maintenance tool (🔷)
# ─────────────────────────────────────────────────────────────────────────────
maintenance = pd.DataFrame(
    [
        ('After COPY / bulk load', 'VACUUM ANALYZE', 'Planner statistics are stale until refreshed'),
        ('Only valid rows queried', 'Partial GiST index', 'Avoid indexing junk geometry and cold rows'),
        ('Repeated range or local scans', 'CLUSTER', 'Neighbouring rows stay close on disk'),
        ('Index exists but never used', 'EXPLAIN + rewrite', 'The predicate may not be index-friendly'),
    ],
    columns=['Situation', 'Tool', 'Reason'],
)
print(maintenance.to_string(index=False))

In [ ]:
# 💻 4.5  Fake before/after planner statistics snapshot (🔷)
# ─────────────────────────────────────────────────────────────────────────────
planner_stats = pd.DataFrame(
    [
        ('Before VACUUM', 'Seq Scan', 500000, 920.3),
        ('After VACUUM', 'Bitmap Heap Scan', 42000, 140.5),
        ('After partial GiST + CLUSTER', 'Index Scan', 6200, 28.7),
    ],
    columns=['Scenario', 'Dominant node', 'Rows visited', 'Actual time (ms)'],
)
print(planner_stats.to_string(index=False))

In [ ]:
# 💻 4.6  Simple maintenance schedule template (🔷)
# ─────────────────────────────────────────────────────────────────────────────
schedule = [
    "Nightly: VACUUM ANALYZE tables changed during the day",
    "Weekly: review pg_stat_user_indexes for unused or underused indexes",
    "After major bulk refresh: rebuild or CLUSTER spatially hot tables",
]
for i, item in enumerate(schedule, start=1):
    print(f"{i}. {item}")

### 📖 Section 4 takeaway

- Fast spatial SQL depends on both query shape and database hygiene.
- `VACUUM ANALYZE` teaches the planner what your table looks like right now; indexes and clustering teach it how to reach the right rows quickly.

---
## 📖 Section 5 — Aggregation: ST_Union, ST_Collect, ST_ConvexHull

Aggregation answers “what does the whole set look like?” but there are several different ways to combine geometries.

| Function | Result | When to use it |
|----------|--------|----------------|
| `ST_Union` | Merged geometry with overlaps dissolved | Final boundaries, coverage zones |
| `ST_Collect` | Multi-geometry collection without dissolving | Fast grouping, later post-processing |
| `ST_ConvexHull` | Smallest convex shell around all features | Quick territory / spread approximation |
| `ST_Envelope` | Bounding rectangle | Fast extents, map framing, coarse filters |

`ST_Union` is usually the most expensive because it changes topology; `ST_Collect` is much cheaper when you only need a grouped container.

In [ ]:
# 💻 5.2  GeoPandas dissolve + Shapely aggregations (🔷)
# ─────────────────────────────────────────────────────────────────────────────
import geopandas as gpd
from shapely.geometry import MultiPoint
from shapely.ops import unary_union

# Use districts + pois from Section 1
# Dissolve all districts into one city boundary
city_boundary = districts.dissolve()
print(f"City boundary area   : {city_boundary.geometry.iloc[0].area:.6f} deg²")
print(f"City boundary type   : {city_boundary.geometry.iloc[0].geom_type}")

# Convex hull of all POIs
poi_multipoint = MultiPoint(list(pois.geometry))
convex_hull    = poi_multipoint.convex_hull
envelope       = poi_multipoint.envelope

print(f"\nPOI convex hull area : {convex_hull.area:.6f} deg²")
print(f"POI envelope area    : {envelope.area:.6f} deg²")
print(f"Hull compactness     : {convex_hull.area / envelope.area:.3f}")

# Dissolve by category
pois_by_cat = gpd.GeoDataFrame(
    {'category': pois['category'], 'geometry': pois.geometry}
).dissolve(by='category', aggfunc='count')
print(f"\nCategory dissolve (count):\n{pois_by_cat}")

In [ ]:
# 💻 5.3  PostGIS aggregation SQL patterns (🐘)
# ─────────────────────────────────────────────────────────────────────────────
sql_lines = [
    "-- Dissolve all district geometries into one city boundary",
    "SELECT ST_Union(geom) AS city_geom FROM districts;",
    "",
    "-- Convex hull of all POI locations",
    "SELECT ST_ConvexHull(ST_Collect(geom)) AS poi_hull FROM pois;",
    "",
    "-- Aggregate POI count + union boundary per category",
    "SELECT",
    "    category,",
    "    COUNT(*)                              AS poi_count,",
    "    ST_ConvexHull(ST_Collect(geom))       AS category_hull,",
    "    ST_Centroid(ST_Collect(geom))         AS category_centroid",
    "FROM pois",
    "GROUP BY category;",
]
print("\n".join(sql_lines))

In [ ]:
# 💻 5.4  Union versus collect versus hull (🔷)
# ─────────────────────────────────────────────────────────────────────────────
from shapely.geometry import GeometryCollection

district_union = unary_union(list(districts.geometry))
district_collect = GeometryCollection(list(districts.geometry))
print(f"Union type   : {district_union.geom_type}")
print(f"Union area   : {district_union.area:.6f}")
print(f"Collect type : {district_collect.geom_type}")
print(f"Collected parts: {len(district_collect.geoms)}")
print(f"Convex hull area of districts: {district_union.convex_hull.area:.6f}")

In [ ]:
# 💻 5.5  Per-category hull and envelope metrics (🔷)
# ─────────────────────────────────────────────────────────────────────────────
rows = []
for category, sub in pois.groupby("category"):
    mp = MultiPoint(list(sub.geometry))
    hull = mp.convex_hull
    env = mp.envelope
    rows.append({
        "category": category,
        "n": len(sub),
        "hull_area": round(hull.area, 6),
        "envelope_area": round(env.area, 6),
    })
print(pd.DataFrame(rows).sort_values("hull_area", ascending=False).to_string(index=False))

### 🎯 Exercise 4 — Largest convex hull by category

**Task:** From the `pois` GeoDataFrame, compute a convex hull for each category and identify which category has the largest hull area.

**Steps:**
1. Group POIs by category.
2. Create a `MultiPoint` for each group and compute `.convex_hull`.
3. Store the hull area and sort descending.

**Hint:**

```python
for category, sub in pois.groupby('category'):
    hull = MultiPoint(list(sub.geometry)).convex_hull
```

In [ ]:
# 🎯 Exercise 4 — your code here ────────────────────────────────
# 1. Group by category.
# 2. Compute each convex hull.
# 3. Print the largest hull area.

In [ ]:
# ✅ Exercise 4 — Solution ──────────────────────────────────────
hull_rows = []
for category, sub in pois.groupby("category"):
    hull = MultiPoint(list(sub.geometry)).convex_hull
    hull_rows.append({"category": category, "hull_area": hull.area})
hull_df = pd.DataFrame(hull_rows).sort_values("hull_area", ascending=False)
print(hull_df.to_string(index=False))
print(f"\nLargest hull: {hull_df.iloc[0]['category']} ({hull_df.iloc[0]['hull_area']:.6f} deg²)")

### 📖 Section 5 takeaway

- Ask whether you need a **merged** geometry (`ST_Union`) or simply a grouped container (`ST_Collect`).
- Hulls and envelopes are fast summary shapes that can be incredibly useful for exploratory analytics and map framing.

---
## 📖 Section 6 — Optimising Slow Spatial Queries

Many spatial queries are slow not because the database is “bad at GIS”, but because the predicate blocks the index or forces unnecessary work.

| Anti-pattern | Fix |
|---|---|
| `ST_Distance(a,b) < N` in `WHERE` | Use `ST_DWithin(a,b,N)` |
| Missing GiST index | `CREATE INDEX ... USING GIST` |
| Spatial function on indexed column in `WHERE` | Add a functional index or rewrite the predicate |
| Large `ST_Union` without simplify | Add `ST_Simplify` before union when acceptable |
| No `VACUUM` after bulk load | Run `VACUUM ANALYZE` after every bulk insert |

In [ ]:
# 💻 6.2  EXPLAIN ANALYZE rewrite pattern (🐘)
# ─────────────────────────────────────────────────────────────────────────────
sql_lines = [
    "-- Slow: ST_Distance in WHERE (no index use)",
    "EXPLAIN ANALYZE",
    "SELECT name FROM urban_features",
    "WHERE ST_Distance(ST_Transform(geom,28992),",
    "    ST_Transform(ST_SetSRID(ST_MakePoint(4.9007,52.3789),4326),28992)) < 500;",
    "",
    "-- Fast: ST_DWithin (uses GiST index)",
    "EXPLAIN ANALYZE",
    "SELECT name FROM urban_features",
    "WHERE ST_DWithin(ST_Transform(geom,28992),",
    "    ST_Transform(ST_SetSRID(ST_MakePoint(4.9007,52.3789),4326),28992), 500);",
]
print("\n".join(sql_lines))

In [ ]:
# 💻 6.3  Benchmark distance filter patterns (🔷)
# ─────────────────────────────────────────────────────────────────────────────
import time
from shapely.geometry import Point
from shapely.strtree import STRtree
import numpy as np

np.random.seed(42)
N = 20_000
pts = [Point(np.random.uniform(4.7,5.1), np.random.uniform(52.2,52.5)) for _ in range(N)]
ref = Point(4.9007, 52.3789)
R   = 0.045  # ~5 km in degrees

# Pattern 1: distance < R for every point (like ST_Distance in WHERE — no index)
t0 = time.perf_counter()
slow_hits = [p for p in pts if p.distance(ref) < R]
t_slow = time.perf_counter() - t0

# Pattern 2: buffer + within (like ST_DWithin — uses index)
tree = STRtree(pts)
buf  = ref.buffer(R)
t0 = time.perf_counter()
fast_candidates = tree.query(buf)
fast_hits = [pts[i] for i in fast_candidates if pts[i].distance(ref) < R]
t_fast = time.perf_counter() - t0

print(f"Pattern 1 (distance<R, no index): {len(slow_hits)} hits in {t_slow*1000:.1f} ms")
print(f"Pattern 2 (buffer+within, index): {len(fast_hits)} hits in {t_fast*1000:.1f} ms")
print(f"Speedup : {t_slow/max(t_fast,0.001):.1f}x")
print("=> Always use ST_DWithin instead of ST_Distance < N in WHERE clauses")

In [ ]:
# 💻 6.4  Slow-query smell table in Python (🔷)
# ─────────────────────────────────────────────────────────────────────────────
slow_query_smells = pd.DataFrame(
    [
        ('distance < N in WHERE', 'ST_DWithin', 'Index can participate'),
        ('No spatial index', 'CREATE INDEX USING GIST', 'Bounding boxes prune candidates'),
        ('Massive union on raw detail', 'Simplify / tile / batch', 'Less topology work at once'),
        ('Repeated transform in predicate', 'Store projected helper geom or functional index', 'Less repeated CPU work'),
    ],
    columns=['Smell', 'Preferred fix', 'Why'],
)
print(slow_query_smells.to_string(index=False))

In [ ]:
# 💻 6.5  Rewrite examples to keep the index useful (🔷)
# ─────────────────────────────────────────────────────────────────────────────
rewrites = [
    ("Bad", "WHERE ST_Distance(a.geom, b.geom) < 500"),
    ("Good", "WHERE ST_DWithin(a.geom, b.geom, 500)"),
    ("Bad", "WHERE ST_Buffer(geom, 100) && ref_geom"),
    ("Better", "WHERE geom && ST_Expand(ref_geom, 100)"),
]
for label, snippet in rewrites:
    print(f"{label:<6} {snippet}")

### 📖 Section 6 takeaway

- The fastest spatial query is usually the one that lets the index reject most rows before exact geometry math starts.
- `ST_DWithin` is a production habit; `ST_Distance` is usually for final measurement or ordering, not first-pass filtering.

---
## 📖 Section 7 — Spatial Aggregation Patterns

Aggregation is how raw feature-level data becomes decision-ready insight: choropleths, district rollups, weighted centroids, and simplified map products all start with grouped spatial statistics.

Typical real-world patterns include heatmaps (grid counts), administrative boundary rollups, population-weighted centres, and dissolve + simplify pipelines for web maps.

In [ ]:
# 💻 7.2  POI density per district choropleth (🔷)
# ─────────────────────────────────────────────────────────────────────────────
import geopandas as gpd
import matplotlib.pyplot as plt

# Reuse districts + pois from Section 1
joined2 = gpd.sjoin(pois, districts[['name','geometry']], how='left', predicate='within')
poi_counts = joined2.groupby('name').size().reset_index(name='poi_count')
districts_enriched = districts.merge(poi_counts, on='name', how='left').fillna(0)

# Choropleth map
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
districts_enriched.plot(column='poi_count', ax=ax, legend=True,
                        cmap='YlOrRd', edgecolor='black',
                        legend_kwds={'label': 'POI count per district'})
pois.plot(ax=ax, color='blue', markersize=8, alpha=0.6, zorder=3)
ax.set_title('POI Density by District', fontsize=14)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig('poi_density_map.png', dpi=100, bbox_inches='tight')
plt.show()
print("Map saved to poi_density_map.png")
print(districts_enriched[['name','poi_count']].to_string(index=False))

In [ ]:
# 💻 7.3  Spatial aggregation SQL patterns (🐘)
# ─────────────────────────────────────────────────────────────────────────────
sql_lines = [
    "-- POI count per district (spatial aggregation)",
    "SELECT",
    "    d.name,",
    "    COUNT(p.id) AS poi_count,",
    "    STRING_AGG(p.category, ', ' ORDER BY p.category) AS categories",
    "FROM districts d",
    "LEFT JOIN pois p ON ST_Within(p.geom, d.geom)",
    "GROUP BY d.id, d.name",
    "ORDER BY poi_count DESC;",
    "",
    "-- Population-weighted centroid",
    "SELECT",
    "    d.name,",
    "    ST_AsText(",
    "        ST_Centroid(",
    "            ST_Union(",
    "                ST_Buffer(ST_Transform(d.geom, 28992), d.population * 0.1)",
    "            )",
    "        )",
    "    ) AS weighted_centroid_rd",
    "FROM districts d",
    "GROUP BY d.name;",
]
print("\n".join(sql_lines))

In [ ]:
# 💻 7.4  Category mix per district (��)
# ─────────────────────────────────────────────────────────────────────────────
mix = joined2.groupby(["name", "category"]).size().unstack(fill_value=0)
mix = mix.reindex(districts["name"], fill_value=0)
mix["total"] = mix.sum(axis=1)
share = mix.div(mix["total"].replace(0, np.nan), axis=0).fillna(0)
print("Counts:")
print(mix.to_string())
print("\nShares:")
print(share.round(2).to_string())

In [ ]:
# 💻 7.5  Most intense district narrative (🔷)
# ─────────────────────────────────────────────────────────────────────────────
district_summary = districts_enriched[["name", "poi_count", "population"]].copy()
district_summary["poi_per_10k_pop"] = (district_summary["poi_count"] / district_summary["population"] * 10_000).round(2)
top_row = district_summary.sort_values("poi_per_10k_pop", ascending=False).iloc[0]
print(district_summary.to_string(index=False))
print(f"\nHighest POI intensity: {top_row['name']} with {top_row['poi_per_10k_pop']} POIs per 10k residents")

---
## 📖 Section 8 — Working with Large Datasets

Large geospatial datasets demand discipline. The goal is to reduce how much data you read, transform, and store at full detail.

- Use chunked reading with pandas `chunksize` to avoid loading huge CSVs at once.
- Use GeoPandas `read_file(..., bbox=...)` or `mask=` to read only the spatial area you need.
- Use PostgreSQL `COPY` for fast bulk loading.
- Simplify complex geometries before serving or storing display-oriented layers.
- Partition big tables by region or time when operational patterns justify it.

In [ ]:
# 💻 8.2  Chunked-style processing and simplification demo (🔷)
# ─────────────────────────────────────────────────────────────────────────────
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
import numpy as np
import time

# Simulate a "large" GeoDataFrame with complex polygons
np.random.seed(0)

def make_complex_polygon(cx, cy, n_vertices=100):
    angles = np.linspace(0, 2*np.pi, n_vertices, endpoint=False)
    radii  = 0.01 + np.random.uniform(-0.003, 0.003, n_vertices)
    xs = cx + radii * np.cos(angles)
    ys = cy + radii * np.sin(angles)
    return Polygon(zip(xs, ys))

complex_polys = [make_complex_polygon(
    np.random.uniform(4.8,5.0), np.random.uniform(52.3,52.5)
) for _ in range(200)]

gdf_complex = gpd.GeoDataFrame(
    {'id': range(200), 'geometry': complex_polys}, crs='EPSG:4326'
)

# Point counts before/after simplify
n_coords_before = sum(len(g.exterior.coords) for g in gdf_complex.geometry)

t0 = time.perf_counter()
gdf_simplified = gdf_complex.copy()
gdf_simplified['geometry'] = gdf_complex.geometry.simplify(0.001, preserve_topology=True)
t_simp = time.perf_counter() - t0

n_coords_after = sum(len(g.exterior.coords) for g in gdf_simplified.geometry)
reduction = (1 - n_coords_after / n_coords_before) * 100

print(f"Polygons         : {len(gdf_complex)}")
print(f"Coords before    : {n_coords_before:,}")
print(f"Coords after     : {n_coords_after:,}")
print(f"Coord reduction  : {reduction:.1f}%")
print(f"Simplify time    : {t_simp*1000:.1f} ms")

In [ ]:
# 💻 8.3  PostGIS bulk loading and simplify SQL (🐘)
# ─────────────────────────────────────────────────────────────────────────────
sql_lines = [
    "-- Fast bulk load with COPY (much faster than INSERT)",
    "COPY urban_features (name, category, geom)",
    "FROM '/data/urban_features.csv'",
    "WITH (FORMAT csv, HEADER true);",
    "",
    "-- Simplify complex geometries for web display",
    "SELECT",
    "    id,",
    "    name,",
    "    ST_SimplifyPreserveTopology(geom, 0.0001) AS geom_simplified,",
    "    ST_NPoints(geom)                           AS pts_before,",
    "    ST_NPoints(ST_SimplifyPreserveTopology(geom, 0.0001)) AS pts_after",
    "FROM urban_features",
    "WHERE ST_GeometryType(geom) IN ('ST_Polygon','ST_MultiPolygon');",
]
print("\n".join(sql_lines))

In [ ]:
# 💻 8.4  Bounding-box filter before deeper work (��)
# ─────────────────────────────────────────────────────────────────────────────
bbox = (4.85, 52.35, 4.95, 52.45)
subset = gdf_complex.cx[bbox[0]:bbox[2], bbox[1]:bbox[3]]
subset_simplified = gdf_simplified.loc[subset.index]
print(f"Total polygons : {len(gdf_complex)}")
print(f"BBox subset    : {len(subset)}")
print(f"Subset share   : {len(subset) / len(gdf_complex):.1%}")
print(f"Subset coords before: {sum(len(g.exterior.coords) for g in subset.geometry):,}")
print(f"Subset coords after : {sum(len(g.exterior.coords) for g in subset_simplified.geometry):,}")

In [ ]:
# 💻 8.5  Chunk-processing recipe sketch (🔷)
# ─────────────────────────────────────────────────────────────────────────────
chunk_edges = np.array_split(gdf_complex.index.to_numpy(), 4)
chunk_rows = []
for chunk_id, idxs in enumerate(chunk_edges, start=1):
    chunk = gdf_complex.loc[idxs]
    chunk_rows.append({
        "chunk": chunk_id,
        "rows": len(chunk),
        "mean_area": round(float(chunk.area.mean()), 6),
    })
print(pd.DataFrame(chunk_rows).to_string(index=False))

### 🎯 Exercise 5 — Flag polygons whose area changed strongly after simplification

**Task:** From `gdf_complex`, find all simplified polygons whose area changed by more than 10% after simplification and print a summary.

**Steps:**
1. Compare original and simplified area row by row.
2. Compute percent change as `abs(after-before)/before * 100`.
3. Filter rows above 10% and summarise how many there are.

**Hint:**

```python
area_before = gdf_complex.geometry.area
area_after = gdf_simplified.geometry.area
```

In [ ]:
# 🎯 Exercise 5 — your code here ────────────────────────────────
# 1. Compute original and simplified area.
# 2. Calculate percent change.
# 3. Filter rows above 10% and print the result.

In [ ]:
# ✅ Exercise 5 — Solution ──────────────────────────────────────
area_before = gdf_complex.geometry.area
area_after = gdf_simplified.geometry.area
pct_change = ((area_after - area_before).abs() / area_before.replace(0, np.nan) * 100).fillna(0)
summary = pd.DataFrame({
    "id": gdf_complex["id"],
    "area_before": area_before.round(6),
    "area_after": area_after.round(6),
    "pct_change": pct_change.round(2),
})
changed = summary[summary["pct_change"] > 10].sort_values("pct_change", ascending=False)
print(changed.head(10).to_string(index=False))
print(f"\nPolygons with >10% area change: {len(changed)} of {len(summary)}")

### 📖 Section 8 takeaway

- Scale problems are often solved by reading less, simplifying early, and batching work instead of trying to brute-force the full dataset at once.
- In PostGIS, `COPY`, bounding-box filters, and simplified derivatives are the database equivalents of chunking and prefiltering in Python.

---
## 📖 Section 9 — Putting It Together: A Spatial Analysis Pipeline

Real projects combine many ideas in sequence: ingest raw data, validate geometry, reproject for metric work, join to reference boundaries, aggregate results, simplify if needed, and export a deliverable.

A reliable pipeline is explicit about **data quality**, **CRS**, and **final output format** at every step.

In [ ]:
# 💻 9.2  End-to-end spatial analysis pipeline (🔷)
# ─────────────────────────────────────────────────────────────────────────────
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from shapely.ops import transform
from shapely.validation import make_valid
from pyproj import Transformer
from pathlib import Path
import json, time

DATA_DIR = Path.home() / '.geospatial_course' / 'week08'
DATA_DIR.mkdir(parents=True, exist_ok=True)

pipeline_start = time.perf_counter()

# Step 1: Synthetic raw data (30 POIs)
import numpy as np
np.random.seed(2024)
raw = gpd.GeoDataFrame({
    'name'    : [f'Site_{i:03d}' for i in range(30)],
    'category': np.random.choice(['school','hospital','park','station'], 30),
    'geometry': [Point(np.random.uniform(4.75,5.05), np.random.uniform(52.30,52.48)) for _ in range(30)]
}, crs='EPSG:4326')

# Step 2: Validate
raw['is_valid'] = raw.geometry.is_valid
n_invalid = (~raw['is_valid']).sum()
if n_invalid:
    raw.geometry = raw.geometry.apply(lambda g: make_valid(g) if not g.is_valid else g)
print(f"Step 2: {n_invalid} invalid geometries repaired")

# Step 3: Reproject to RD New
raw_rd = raw.to_crs('EPSG:28992')
print(f"Step 3: Reprojected to EPSG:28992")

# Step 4: Spatial join to districts (reproject districts too)
dist_rd = districts.to_crs('EPSG:28992')
joined_rd = gpd.sjoin(raw_rd, dist_rd[['name','population','geometry']], how='left', predicate='within')
joined_rd.rename(columns={'name_right':'district','name_left':'site_name'}, inplace=True)

# Step 5: Aggregate
agg = joined_rd.groupby(['district','category']).size().reset_index(name='count')
print(f"Step 4-5: Spatial join + aggregation complete")
print(agg.to_string(index=False))

# Step 6: Export GeoJSON
out_path = DATA_DIR / 'pipeline_output.geojson'
joined_rd.to_crs('EPSG:4326').to_file(str(out_path), driver='GeoJSON')
elapsed = time.perf_counter() - pipeline_start

# Step 7: Report
report = {
    'total_sites'    : len(raw),
    'invalid_repaired': n_invalid,
    'unmatched'      : int(joined_rd['district'].isna().sum()),
    'districts_hit'  : int(joined_rd['district'].notna().nunique()),
    'output_file'    : str(out_path),
    'elapsed_s'      : round(elapsed, 3),
}
print(f"\nPipeline report:\n{json.dumps(report, indent=2)}")

In [ ]:
# 💻 9.3  Full PostGIS pipeline SQL sketch (🐘)
# ─────────────────────────────────────────────────────────────────────────────
sql_lines = [
    "-- 1) Stage raw records",
    "CREATE TEMP TABLE stage_sites (",
    "    name TEXT,",
    "    category TEXT,",
    "    geom geometry(Point, 4326)",
    ");",
    "",
    "-- 2) Insert or COPY staged data",
    "INSERT INTO stage_sites (name, category, geom)",
    "SELECT name, category, geom FROM incoming_sites;",
    "",
    "-- 3) Repair invalid geometry",
    "UPDATE stage_sites",
    "SET geom = ST_MakeValid(geom)",
    "WHERE NOT ST_IsValid(geom);",
    "",
    "-- 4) Index staged data for joins",
    "CREATE INDEX stage_sites_geom_idx ON stage_sites USING GIST (geom);",
    "ANALYZE stage_sites;",
    "",
    "-- 5) Build result table with join + aggregation",
    "INSERT INTO district_site_summary (district_name, category, site_count)",
    "SELECT d.name, s.category, COUNT(*)",
    "FROM districts d",
    "JOIN stage_sites s ON ST_Within(s.geom, d.geom)",
    "GROUP BY d.name, s.category;",
    "",
    "-- 6) Maintenance after write-heavy work",
    "VACUUM ANALYZE district_site_summary;",
    "",
    "-- 7) Final report",
    "SELECT district_name, SUM(site_count) AS total_sites",
    "FROM district_site_summary",
    "GROUP BY district_name",
    "ORDER BY total_sites DESC;",
]
print("\n".join(sql_lines))

In [ ]:
# 💻 9.4  Pipeline QA checks (🔷)
# ─────────────────────────────────────────────────────────────────────────────
qa = {
    "total_rows": len(joined_rd),
    "unmatched": int(joined_rd["district"].isna().sum()),
    "categories": sorted(raw["category"].unique().tolist()),
    "districts_hit": sorted(joined_rd["district"].dropna().unique().tolist()),
}
print(json.dumps(qa, indent=2))

In [ ]:
# 💻 9.5  Reload exported GeoJSON and inspect schema (🔷)
# ─────────────────────────────────────────────────────────────────────────────
reloaded = gpd.read_file(out_path)
print(reloaded.head(5).to_string())
print(f"\nRows: {len(reloaded)} | CRS: {reloaded.crs}")
print(f"Columns: {list(reloaded.columns)}")

## 🔬 Mini-Lab — Optimise and Analyse

**Scenario:** You are a performance-focused spatial analyst. You have a dataset of 30 sites and 5 districts. Your task is to run a complete analysis, check geometry validity, optimise the query approach, compute nearest neighbours, aggregate by district, and export a final report.

In [ ]:
# 🔬 Step 1 — Data prep
# ─────────────────────────────────────────────────────────────────────────────
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point, Polygon

lab_districts = gpd.GeoDataFrame(
    {
        'name': ['North', 'South', 'East', 'West', 'Centre'],
        'population': [45000, 62000, 38000, 51000, 89000],
        'geometry': [
            Polygon([(4.85,52.40),(4.95,52.40),(4.95,52.47),(4.85,52.47),(4.85,52.40)]),
            Polygon([(4.85,52.33),(4.95,52.33),(4.95,52.40),(4.85,52.40),(4.85,52.33)]),
            Polygon([(4.95,52.35),(5.05,52.35),(5.05,52.45),(4.95,52.45),(4.95,52.35)]),
            Polygon([(4.75,52.35),(4.85,52.35),(4.85,52.45),(4.75,52.45),(4.75,52.35)]),
            Polygon([(4.87,52.36),(4.93,52.36),(4.93,52.42),(4.87,52.42),(4.87,52.36)]),
        ],
    },
    crs='EPSG:4326'
)
np.random.seed(2024)
lab_sites = gpd.GeoDataFrame(
    {
        'site_name': [f'Site_{i:03d}' for i in range(30)],
        'category': np.random.choice(['school','hospital','park','station'], 30),
        'geometry': [Point(np.random.uniform(4.75,5.05), np.random.uniform(52.30,52.48)) for _ in range(30)]
    },
    crs='EPSG:4326'
)
print(f"Districts shape: {lab_districts.shape} | CRS: {lab_districts.crs}")
print(f"Sites shape    : {lab_sites.shape} | CRS: {lab_sites.crs}")

In [ ]:
# 🔬 Step 2 — Validity audit
# ─────────────────────────────────────────────────────────────────────────────
from shapely.validation import explain_validity, make_valid

for label, gdf in [("districts", lab_districts), ("sites", lab_sites)]:
    invalid_mask = ~gdf.geometry.is_valid
    invalid_count = int(invalid_mask.sum())
    print(f"{label:<9} invalid geometries: {invalid_count}")
    if invalid_count:
        gdf.loc[invalid_mask, "geometry"] = gdf.loc[invalid_mask, "geometry"].apply(make_valid)
        print(gdf.loc[invalid_mask, [c for c in gdf.columns if c != "geometry"]].to_string(index=False))
    else:
        print(f"  All {len(gdf)} geometries valid")

In [ ]:
# 🔬 Step 3 — Nearest-neighbour table
# ─────────────────────────────────────────────────────────────────────────────
lab_sites_rd = lab_sites.to_crs(28992)
lab_districts_rd = lab_districts.to_crs(28992)
lab_centroids_rd = lab_districts_rd.copy()
lab_centroids_rd["centroid"] = lab_centroids_rd.geometry.centroid
rows = []
for _, srow in lab_sites_rd.iterrows():
    dists = lab_centroids_rd["centroid"].distance(srow.geometry)
    idx = dists.idxmin()
    rows.append({
        "site_name": lab_sites.loc[srow.name, "site_name"],
        "category": lab_sites.loc[srow.name, "category"],
        "nearest_district": lab_districts.loc[idx, "name"],
        "dist_m": round(float(dists.loc[idx]), 1),
    })
lab_nearest = pd.DataFrame(rows).sort_values("dist_m")
print(lab_nearest.head(10).to_string(index=False))

In [ ]:
# 🔬 Step 4 — Optimised spatial join
# ─────────────────────────────────────────────────────────────────────────────
lab_joined = gpd.sjoin(lab_sites, lab_districts[['name','population','geometry']], how='left', predicate='within')
lab_joined.rename(columns={'name_right':'district','site_name':'site_name'}, inplace=True)
unmatched_mask = lab_joined["district"].isna()
print(f"Unmatched after within join: {int(unmatched_mask.sum())}")
if unmatched_mask.any():
    centroids_rd = lab_districts.to_crs(28992).geometry.centroid
    sites_rd = lab_sites.to_crs(28992)
    for idx in lab_joined.index[unmatched_mask]:
        dists = centroids_rd.distance(sites_rd.loc[idx, "geometry"])
        nearest_idx = dists.idxmin()
        lab_joined.loc[idx, "district"] = lab_districts.loc[nearest_idx, "name"]
        lab_joined.loc[idx, "population"] = lab_districts.loc[nearest_idx, "population"]
print(lab_joined[["site_name", "category", "district"]].head(10).to_string(index=False))

In [ ]:
# 🔬 Step 5 — Aggregate statistics
# ─────────────────────────────────────────────────────────────────────────────
lab_joined["x_coord"] = lab_joined.geometry.x
lab_summary = lab_joined.groupby(["district", "category"]).agg(
    site_count=("site_name", "count"),
    mean_x=("x_coord", "mean"),
).reset_index()
lab_summary["mean_x"] = lab_summary["mean_x"].round(5)
print(lab_summary.sort_values(["district", "category"]).to_string(index=False))

In [ ]:
# 🔬 Step 6 — Export + summary
# ─────────────────────────────────────────────────────────────────────────────
from pathlib import Path
import json

LAB_DIR = Path.home() / '.geospatial_course' / 'week08_lab'
LAB_DIR.mkdir(parents=True, exist_ok=True)
lab_out = LAB_DIR / 'week08_lab_output.geojson'
lab_joined.to_file(lab_out, driver="GeoJSON")
lab_report = {
    "total_sites": int(len(lab_joined)),
    "districts": int(lab_joined["district"].nunique()),
    "categories": sorted(lab_joined["category"].unique().tolist()),
    "unmatched_count": int(lab_joined["district"].isna().sum()),
    "output_file": str(lab_out),
}
print(json.dumps(lab_report, indent=2))

## 🏁 Week 8 Summary

| Section | Key Concept |
|---------|-------------|
| 1 | Spatial joins: gpd.sjoin() + PostGIS LATERAL join |
| 2 | KNN: STRtree nearest + PostGIS <-> operator |
| 3 | Geometry validity: explain_validity, make_valid, buffer(0) |
| 4 | Performance: VACUUM ANALYZE, partial indexes, CLUSTER |
| 5 | Aggregation: ST_Union, ST_Collect, ST_ConvexHull |
| 6 | Query optimisation: ST_DWithin beats ST_Distance < N |
| 7 | Density maps: sjoin + groupby + choropleth |
| 8 | Large datasets: chunking, simplify, COPY bulk load |
| 9 | End-to-end pipeline: load→validate→reproject→join→export |
| 10 | Mini-Lab: optimise, analyse, nearest-neighbour, export |

### ☑️ Self-assessment checklist
- [ ] I can perform a spatial join with gpd.sjoin() and handle unmatched rows
- [ ] I understand KNN with STRtree and the PostGIS <-> operator
- [ ] I can detect, explain, and repair invalid geometries
- [ ] I know when to use VACUUM ANALYZE and partial GiST indexes
- [ ] I can dissolve geometries with ST_Union and collect with ST_Collect
- [ ] I understand why ST_DWithin is faster than ST_Distance < N
- [ ] I can build a choropleth density map from a spatial join
- [ ] I can simplify geometries and measure coordinate reduction
- [ ] I can build an end-to-end spatial analysis pipeline
- [ ] I am ready to tackle CityJSON 3D data in Week 9

## 📚 Week 9 Preview

| Topic | What you'll learn |
|-------|-------------------|
| CityJSON schema | JSON structure, versioning, metadata |
| Object types | Building, Road, WaterBody, CityFurniture, LandUse |
| Geometry arrays | vertices list, geometry templates, semantics |
| Python parsing | cityjson library, manual json.load() approach |
| Practical extraction | Building footprints, heights, attributes from CityJSON |

## 📖 Further Reading

| Resource | Link |
|----------|------|
| GeoPandas sjoin docs | https://geopandas.org/en/stable/docs/reference/api/geopandas.sjoin.html |
| PostGIS KNN | https://postgis.net/workshops/postgis-intro/knn.html |
| Shapely validation | https://shapely.readthedocs.io/en/stable/validation.html |
| PostGIS performance | https://postgis.net/docs/performance_tips.html |
| ST_SimplifyPreserveTopology | https://postgis.net/docs/ST_SimplifyPreserveTopology.html |

---
*Next: **Week 9 — CityJSON Parsing & 3D Geospatial Data** — parse building footprints and extract metadata from 3D city models.*